# Claude Structured-Output Evaluation

Evaluate a selected Claude model on Calculus I multiple-choice questions. Responses use Claude's native structured-output API and are checkpointed after each successful question so interrupted runs can resume.

## 1. Imports and configuration

In [ ]:
import json
import os
import re
from pathlib import Path

import pandas as pd
from anthropic import Anthropic
from dotenv import load_dotenv
from pydantic import BaseModel
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_recall_fscore_support,
)

### Run configuration

Change only `MODEL_NAME` to test another Claude model. The artifact directory and every generated filename update automatically.

In [ ]:
# Support launching Jupyter from either the repository root or notebooks/.
WORKING_DIRECTORY = Path.cwd()
PROJECT_ROOT = (
    WORKING_DIRECTORY.parent
    if WORKING_DIRECTORY.name == "notebooks"
    else WORKING_DIRECTORY
)

MODEL_NAME = "claude-opus-4-5-20251101"
RUN_NAME = f"{MODEL_NAME}-subsample"

DATA_PATH = PROJECT_ROOT / "data" / "input_data.csv"
MODEL_OUTPUT_DIRECTORY = PROJECT_ROOT / "output" / MODEL_NAME
CHECKPOINT_PATH = MODEL_OUTPUT_DIRECTORY / f"{RUN_NAME}-checkpoint.json"
RESULTS_PATH = MODEL_OUTPUT_DIRECTORY / f"{RUN_NAME}-test-output.csv"
INCORRECT_RESULTS_PATH = (
    MODEL_OUTPUT_DIRECTORY / f"{RUN_NAME}-test-output-incorrect.csv"
)

MODEL_OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

## 2. Provider clients and response schemas

In [ ]:
load_dotenv(PROJECT_ROOT / ".env")
client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))


class Calculus(BaseModel):
    """Structured response for one calculus question."""

    detailed_solution_explanation: str
    correct_option_choice_letter: str

## 3. Load and inspect the dataset

In [ ]:
def load_dataset(data_path: Path) -> pd.DataFrame:
    """Load the evaluation dataset."""
    return pd.read_csv(data_path, keep_default_na=False)


def summarize_dataset(dataframe: pd.DataFrame) -> pd.Series:
    """Summarize row counts and duplicate checks used in the original notebook."""
    return pd.Series(
        {
            "rows": len(dataframe),
            "columns": len(dataframe.columns),
            "duplicate_rows": int(dataframe.duplicated().sum()),
            "duplicate_questions": int(dataframe["Question"].duplicated().sum()),
            "duplicate_ids": int(dataframe["ID"].duplicated().sum()),
        }
    )


df = load_dataset(DATA_PATH)
print(f"Loaded {len(df)} questions from {DATA_PATH}")
summarize_dataset(df)

In [ ]:
# Inspect any duplicated rows, questions, or IDs found by the summary above.
duplicate_rows = df.loc[df.duplicated(keep=False)].sort_values(df.columns.tolist())
duplicate_questions = df.loc[df["Question"].duplicated(keep=False)].sort_values(
    "Question"
)
duplicate_ids = df.loc[df["ID"].duplicated(keep=False)].sort_values("ID")

df.describe(include="all")

## 4. Primary-model response pipeline

### Few-shot example and system instructions

In [ ]:
EXAMPLE_RESPONSE = """Question:
Complete the square to write the equation in the standard form f(x)=a(x-h)^2+k.
f(x)=2x^2+12x+25
    a) f(x)=2(x+3)^2+16
    b) f(x)=2(x-6)^2+7
    c) f(x)=2(x+3)^2+7
    d) f(x)=2(x-3)^2+16
    e) f(x)=2(x+3)^2-7
    f) None of the above.

Solution:
f(x) = 2(x^2 + 6x) + 25
     = 2((x + 3)^2 - 9) + 25
     = 2(x + 3)^2 + 7.
The calculated solution matches option c.
Final answer: c
"""

SYSTEM_PROMPT = (
    "Act as a mathematician specializing in Calculus. Solve the given "
    "college-level Calculus I multiple-choice problem using the example as a "
    "reference. Provide a clear, accurate, step-by-step explanation of the "
    "solution. Compare the calculated solution with the provided answer "
    "choices and output the matching option letter. If no answer choice "
    'matches the calculated solution, output "NA" as the final answer. '
    'Do not force a match or choose "None of the above" instead of "NA" '
    "when no listed solution matches.\n\nHere is the example:\n"
    + EXAMPLE_RESPONSE
)

RESPONSE_FORMAT_INSTRUCTIONS = (
    "Return only one valid JSON object with the keys "
    "detailed_solution_explanation and correct_option_choice_letter. "
    'Set correct_option_choice_letter to the matching option letter or "NA". '
    "Do not wrap the JSON in Markdown."
)


In [ ]:
def load_checkpoint(checkpoint_path: Path) -> dict:
    """Load saved responses, or return an empty checkpoint."""
    if not checkpoint_path.exists():
        return {}
    with checkpoint_path.open("r", encoding="utf-8") as checkpoint_file:
        return json.load(checkpoint_file)


def save_checkpoint(checkpoint_path: Path, data: dict) -> None:
    """Persist responses as formatted JSON."""
    with checkpoint_path.open("w", encoding="utf-8") as checkpoint_file:
        json.dump(data, checkpoint_file, indent=4)

In [ ]:
def query_claude(question: str, model: str = MODEL_NAME) -> dict:
    """Request one structured calculus solution from Claude."""
    try:
        response = client.messages.parse(
            model=model,
            max_tokens=4096,
            system=SYSTEM_PROMPT + "\n\n" + RESPONSE_FORMAT_INSTRUCTIONS,
            messages=[
                {
                    "role": "user",
                    "content": "Now solve the following question: " + question,
                }
            ],
            output_format=Calculus,
        )
    except Exception as error:
        print(f"Error querying Claude: {type(error).__name__}")
        return {}

    parsed_output = response.parsed_output
    return parsed_output.model_dump() if parsed_output else {}

In [ ]:
def collect_responses(
    dataframe: pd.DataFrame,
    checkpoint_path: Path,
    model: str = MODEL_NAME,
) -> dict:
    """Query unanswered rows and update the checkpoint incrementally."""
    checkpoint = load_checkpoint(checkpoint_path)

    for _, row in dataframe.iterrows():
        question_id = str(row["ID"])
        if question_id in checkpoint:
            print(f"Skipping ID {question_id}, already processed.")
            continue

        print(f"Processing ID {question_id}...")
        response = query_claude(row["Question"], model=model)
        if not response:
            print(f"Failed to get response for {question_id}")
            continue

        checkpoint[question_id] = response
        save_checkpoint(checkpoint_path, checkpoint)

    save_checkpoint(checkpoint_path, checkpoint)
    print(f"Checkpoint saved to {checkpoint_path}")
    return checkpoint

### Run the primary-model pipeline

> **Note:** This cell makes billable Anthropic API calls only for questions absent from the checkpoint.

In [ ]:
df = df.head(10)  # Limit to first 10 rows for testing

checkpoint = collect_responses(df, CHECKPOINT_PATH)
df.head(10)

## 5. Prepare answer choices

Read the structured answer fields already saved by the primary model; no extraction-model call is needed.

In [ ]:
def add_model_responses(dataframe: pd.DataFrame, checkpoint: dict) -> pd.DataFrame:
    """Add answer choices and explanations from the checkpoint."""
    result = dataframe.copy()
    question_ids = result["ID"].astype(str)
    result["LLM Answer Choice (RAW)"] = question_ids.map(
        lambda question_id: checkpoint.get(question_id, {}).get(
            "correct_option_choice_letter", ""
        )
    )
    result["LLM Answer Explanation"] = question_ids.map(
        lambda question_id: checkpoint.get(question_id, {}).get(
            "detailed_solution_explanation", ""
        )
    )
    return result

### Prepare answer choices from the checkpoint

In [ ]:
df = add_model_responses(df, checkpoint)
df.head()

## 6. Normalize and evaluate answers

In [ ]:
def extract_letter_choice(raw_response: str) -> str:
    """Normalize a letter or NA from plain text or an extraction JSON object."""
    if not isinstance(raw_response, str):
        return ""
    value = raw_response.strip()
    if value.upper() == "NA":
        return "NA"
    if re.fullmatch(r"[a-zA-Z]", value):
        return value.lower()
    try:
        payload = json.loads(value)
    except json.JSONDecodeError:
        return ""
    if not isinstance(payload, dict):
        return ""
    choice = payload.get("correct_option_choice_letter")
    if not isinstance(choice, str):
        return ""
    choice = choice.strip()
    if choice.upper() == "NA":
        return "NA"
    return choice.lower() if re.fullmatch(r"[a-zA-Z]", choice) else ""


def evaluate_answers(dataframe: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Normalize predictions and calculate weighted classification metrics."""
    result = dataframe.copy()
    result["LLM Answer Choice"] = result["LLM Answer Choice (RAW)"].map(
        extract_letter_choice
    )
    result["Correct"] = (
        result["Correct Answer"] == result["LLM Answer Choice"]
    ).astype(int)

    y_true = result["Correct Answer"]
    y_pred = result["LLM Answer Choice"]
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "classification_report": classification_report(
            y_true, y_pred, zero_division=0
        ),
    }
    return result, metrics

In [ ]:
df, metrics = evaluate_answers(df)

print(metrics["classification_report"])
print(f"Accuracy:  {metrics['accuracy'] * 100:.2f}%")
print(f"Precision: {metrics['precision'] * 100:.2f}%")
print(f"Recall:    {metrics['recall'] * 100:.2f}%")
print(f"F1 Score:  {metrics['f1'] * 100:.2f}%")
df.head()

## 7. Export results

In [ ]:
incorrect_df = df.loc[df["Correct"] == 0].copy()

incorrect_df.to_csv(INCORRECT_RESULTS_PATH, index=False)
df.to_csv(RESULTS_PATH, index=False)

print(f"Saved {len(df)} results to {RESULTS_PATH}")
print(f"Saved {len(incorrect_df)} incorrect results to {INCORRECT_RESULTS_PATH}")